Nassau Candy Distributor
FactorytoCustomer Shipping Route Efficiency Analysis
Google Colab Notebook | Dataset: Nassau_Candy.csv | Period: Jan 2024 Dec 2025
Instructions: Run each cell top to bottom. When prompted, upload your Nassau_Candy.csv file.


In [ ]:
Cell 1: Install dependencies
!pip install plotly folium quiet
print(" Dependencies ready.")


In [ ]:
Cell 2: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from google.colab import files
import warnings
warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 5)
sns.set_theme(style="whitegrid", palette="muted")
print(" Libraries loaded successfully.")


Step 1: Upload Your Data


In [ ]:
Cell 3: Upload & Load Data
print(" Please upload your Nassau_Candy.csv file:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df_raw = pd.read_csv(filename)
print(f" Loaded {len(df_raw):,} rows {df_raw.shape[1]} columns")
df_raw.head()


Step 2: Data Cleaning & Validation


In [ ]:
Cell 4: Data Cleaning & Validation
df = df_raw.copy()
Parse dates (DDMMYYYY format)
df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True, errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], dayfirst=True, errors="coerce")
Drop rows with unparseable dates
before = len(df)
df.dropna(subset=["Order Date", "Ship Date"], inplace=True)
print(f" Rows dropped (bad dates): {before len(df)}")
Calculate Shipping Lead Time
df["Lead Time"] = (df["Ship Date"] df["Order Date"]).dt.days
Remove invalid (negative) lead times
neg = (df["Lead Time"] < 0).sum()
df = df[df["Lead Time"] = 0]
print(f" Rows dropped (negative lead time): {neg}")
Standardise text fields
for col in ["Ship Mode", "Division", "Region", "State/Province", "City"]:
df[col] = df[col].str.strip().str.title()
Handle missing values
df["Sales"].fillna(0, inplace=True)
df["Gross Profit"].fillna(0, inplace=True)
df["Units"].fillna(0, inplace=True)
print(f"\n Clean dataset: {len(df):,} rows")
df.dtypes


Step 3: Feature Engineering


In [ ]:
Cell 5: Feature Engineering
Factory mapping by Product Name
factory_map = {
"Wonka Bar Nutty Crunch Surprise" : "Lot's O Nuts",
"Wonka Bar Fudge Mallows" : "Lot's O Nuts",
"Wonka Bar Scrumdiddlyumptious" : "Lot's O Nuts",
"Wonka Bar Milk Chocolate" : "Wicked Choccy's",
"Wonka Bar Triple Dazzle Caramel" : "Wicked Choccy's",
"Laffy Taffy" : "Sugar Shack",
"Sweetarts" : "Sugar Shack",
"Nerds" : "Sugar Shack",
"Fun Dip" : "Sugar Shack",
"Fizzy Lifting Drinks" : "Sugar Shack",
"Everlasting Gobstopper" : "Secret Factory",
"Hair Toffee" : "The Other Factory",
"Lickable Wallpaper" : "Secret Factory",
"Wonka Gum" : "Secret Factory",
"Kazookles" : "The Other Factory",
}
df["Factory"] = df["Product Name"].map(factory_map).fillna("Unknown")
Factory coordinates
factory_coords = {
"Lot's O Nuts" : (32.881893, 111.768036),
"Wicked Choccy's" : (32.076176, 81.088371),
"Sugar Shack" : (48.119140, 96.181150),
"Secret Factory" : (41.446333, 90.565487),
"The Other Factory": (35.117500, 89.971107),
}
df["Factory Lat"] = df["Factory"].map(lambda f: factory_coords.get(f, (None, None))[0])
df["Factory Lon"] = df["Factory"].map(lambda f: factory_coords.get(f, (None, None))[1])
Route identifiers
df["Route"] = df["Factory"] + " " + df["State/Province"]
df["Route (Region)"]= df["Factory"] + " " + df["Region"]
print(" Feature engineering complete.")
df[["Factory", "Route", "Lead Time"]].head(8)


Step 4: KPI Summary


In [ ]:
Cell 6: KPI Summary
print("=" 55)
print(" KEY PERFORMANCE INDICATORS (KPIs)")
print("=" 55)
print(f" Total Orders : {len(df):10,}")
print(f" Total Sales : ${df['Sales'].sum():10,.2f}")
print(f" Total Gross Profit : ${df['Gross Profit'].sum():10,.2f}")
print(f" Total Units Shipped : {df['Units'].sum():10,}")
print(f" Avg Shipping Lead Time : {df['Lead Time'].mean():10.2f} days")
print(f" Median Lead Time : {df['Lead Time'].median():10.2f} days")
print(f" Max Lead Time : {df['Lead Time'].max():10} days")
print(f" Min Lead Time : {df['Lead Time'].min():10} days")
print("=" 55)


Step 5: Route Efficiency Analysis


In [ ]:
Cell 7: Route Efficiency Analysis
route_stats = (
df.groupby("Route")
.agg(
Total_Shipments = ("Lead Time", "count"),
Avg_Lead_Time = ("Lead Time", "mean"),
Std_Lead_Time = ("Lead Time", "std"),
Total_Sales = ("Sales", "sum"),
Total_Profit = ("Gross Profit", "sum"),
)
.round(2)
.reset_index()
)
Efficiency Score: lower avg lead time & variability = higher score
max_lt = route_stats["Avg_Lead_Time"].max()
max_std = route_stats["Std_Lead_Time"].max()
route_stats["Efficiency Score"] = (
1 (route_stats["Avg_Lead_Time"] / max_lt) 0.7
(route_stats["Std_Lead_Time"].fillna(0) / (max_std + 1)) 0.3
).round(4)
route_stats.sort_values("Efficiency Score", ascending=False, inplace=True)
route_stats.reset_index(drop=True, inplace=True)
print(" Top 10 Most Efficient Routes:")
display(route_stats.head(10)[["Route","Avg_Lead_Time","Total_Shipments","Efficiency Score"]])
print("\n Bottom 10 Least Efficient Routes:")
display(route_stats.tail(10)[["Route","Avg_Lead_Time","Total_Shipments","Efficiency Score"]])


Step 6: Ship Mode Performance


In [ ]:
Cell 8: Ship Mode Performance
ship_mode = df.groupby("Ship Mode").agg(
Count = ("Lead Time", "count"),
Avg_LT = ("Lead Time", "mean"),
Med_LT = ("Lead Time", "median"),
Avg_Cost = ("Cost", "mean"),
).round(2).reset_index().sort_values("Avg_LT")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["4CAF50","2196F3","FF9800","F44336"]
axes[0].barh(ship_mode["Ship Mode"], ship_mode["Avg_LT"], color=colors)
axes[0].set_xlabel("Average Lead Time (days)")
axes[0].set_title("Avg Lead Time by Ship Mode")
for i, v in enumerate(ship_mode["Avg_LT"]):
axes[0].text(v + 0.1, i, f"{v:.1f}d", va="center", fontsize=10)
axes[1].barh(ship_mode["Ship Mode"], ship_mode["Count"], color=colors)
axes[1].set_xlabel("Number of Shipments")
axes[1].set_title("Shipment Volume by Ship Mode")
plt.tight_layout()
plt.savefig("ship_mode_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
display(ship_mode)


Step 7: Regional & State Bottleneck Analysis


In [ ]:
Cell 9: Regional Analysis
region_perf = df.groupby("Region").agg(
Avg_LT = ("Lead Time", "mean"),
Count = ("Lead Time", "count"),
Total_Sales= ("Sales", "sum"),
).round(2).reset_index().sort_values("Avg_LT", ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(region_perf["Region"], region_perf["Avg_LT"],
color=["F44336","FF9800","4CAF50","2196F3"])
ax.set_ylabel("Average Lead Time (days)")
ax.set_title("Regional Shipping Performance Avg Lead Time")
ax.axhline(df["Lead Time"].mean(), color="gray", linestyle="",
label=f"Overall Avg: {df['Lead Time'].mean():.1f}d")
ax.legend()
for bar, val in zip(bars, region_perf["Avg_LT"]):
ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
f"{val:.1f}d", ha="center", fontsize=10)
plt.tight_layout()
plt.savefig("regional_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
display(region_perf)


In [ ]:
Cell 10: StateLevel Top & Bottom
state_perf = df.groupby("State/Province")["Lead Time"].mean().round(2).reset_index()
state_perf.columns = ["State", "Avg Lead Time"]
top5 = state_perf.nsmallest(5, "Avg Lead Time")
bottom5 = state_perf.nlargest(5, "Avg Lead Time")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(top5["State"], top5["Avg Lead Time"], color="4CAF50")
axes[0].set_title(" Top 5 Fastest States")
axes[0].set_xlabel("Avg Lead Time (days)")
for i, v in enumerate(top5["Avg Lead Time"]):
axes[0].text(v + 0.1, i, f"{v:.1f}d", va="center")
axes[1].barh(bottom5["State"], bottom5["Avg Lead Time"], color="F44336")
axes[1].set_title(" Top 5 Slowest States")
axes[1].set_xlabel("Avg Lead Time (days)")
for i, v in enumerate(bottom5["Avg Lead Time"]):
axes[1].text(v + 0.1, i, f"{v:.1f}d", va="center")
plt.tight_layout()
plt.savefig("state_performance.png", dpi=150, bbox_inches="tight")
plt.show()


Step 8: Division & Product Revenue


In [ ]:
Cell 11: Division & Product Sales
div_sales = df.groupby("Division")[["Sales","Gross Profit"]].sum().round(2)
print("Sales by Division:")
display(div_sales)
top_products = df.groupby("Product Name")["Sales"].sum().sort_values(ascending=False).head(10)
fig = px.bar(top_products.reset_index(), x="Sales", y="Product Name",
orientation="h", title="Top 10 Products by Total Sales",
color="Sales", color_continuous_scale="Blues")
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()


Step 9: Lead Time Distribution & Trends


In [ ]:
Cell 12: Lead Time Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df["Lead Time"], bins=40, color="2196F3", edgecolor="white")
axes[0].set_xlabel("Lead Time (days)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Lead Time Distribution")
df.boxplot(column="Lead Time", by="Ship Mode", ax=axes[1])
axes[1].set_title("Lead Time by Ship Mode")
axes[1].set_xlabel("Ship Mode")
axes[1].set_ylabel("Lead Time (days)")
plt.suptitle("")
plt.tight_layout()
plt.savefig("lead_time_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
Cell 13: Monthly Trend
df["Month"] = df["Order Date"].dt.to_period("M")
monthly = df.groupby("Month").agg(
Orders = ("Lead Time","count"),
Avg_LT = ("Lead Time","mean")
).reset_index()
monthly["Month"] = monthly["Month"].astype(str)
fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
ax1.bar(monthly["Month"], monthly["Orders"], color="90CAF9", label="Order Volume")
ax2.plot(monthly["Month"], monthly["Avg_LT"], color="F44336", marker="o", label="Avg Lead Time")
ax1.set_xlabel("Month")
ax1.set_ylabel("Order Count", color="2196F3")
ax2.set_ylabel("Avg Lead Time (days)", color="F44336")
ax1.set_title("Monthly Order Volume & Avg Lead Time Trend")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("monthly_trend.png", dpi=150, bbox_inches="tight")
plt.show()


Step 10: Factory Performance


In [ ]:
Cell 14: Factory Performance Bubble Chart
factory_perf = df.groupby("Factory").agg(
Shipments = ("Lead Time", "count"),
Avg_LT = ("Lead Time", "mean"),
Sales = ("Sales", "sum"),
).round(2).reset_index().sort_values("Avg_LT")
fig = px.scatter(factory_perf, x="Avg_LT", y="Sales",
size="Shipments", color="Factory", text="Factory",
title="Factory: Avg Lead Time vs Total Sales (bubble size = shipments)",
labels={"Avg_LT": "Avg Lead Time (days)", "Sales": "Total Sales ($)"})
fig.update_traces(textposition="top center")
fig.show()
display(factory_perf)


Step 11: Export Results


In [ ]:
Cell 15: Export Clean Data & Summary
df.to_csv("nassau_candy_clean.csv", index=False)
route_stats.to_csv("route_efficiency_report.csv", index=False)
print(" Exported: nassau_candy_clean.csv")
print(" Exported: route_efficiency_report.csv")
files.download("nassau_candy_clean.csv")
files.download("route_efficiency_report.csv")
print("\n Analysis Complete! All charts saved as PNG files in your Colab session.")
